In [1]:
from datasets import load_dataset
import numpy as np

# There is only one split on the hub
dataset = load_dataset("OGB/ogbg-molhiv", cache_dir="./data")
seed = 42
dataset = dataset
# activate notebook autoload of files
%load_ext autoreload
%autoreload 2

/opt/miniconda3/envs/graphormer_lora/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from transformers import TrainingArguments, GraphormerForGraphClassification

model = GraphormerForGraphClassification.from_pretrained(
    "clefourrier/pcqm4mv2_graphormer_base",
    num_classes=2, # num_classes for the downstream task 
    ignore_mismatched_sizes=True,
)

training_args = TrainingArguments(
    "graph-classification/test/molhiv",
    logging_dir="graph-classification",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    auto_find_batch_size=False,
    gradient_accumulation_steps=4,
    dataloader_num_workers=8,
    num_train_epochs=4,
    evaluation_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    push_to_hub=False,
    dataloader_drop_last=True,
    remove_unused_columns=False,  # CRITICAL: Don't remove columns - GraphormerDataCollator needs them!
    seed=seed,
    use_mps_device=True,  # Force MPS usage for both training and evaluation
    dataloader_prefetch_factor=2,
    include_inputs_for_metrics=False)
model.to("mps")

/opt/miniconda3/envs/graphormer_lora/lib/python3.11/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of GraphormerForGraphClassification were not initialized from the model checkpoint at clefourrier/pcqm4mv2_graphormer_base and are newly initialized because the shapes did not match:
- classifier.classifier.weight: found shape torch.Size([1, 768]) in the checkpoint and torch.Size([2, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/opt/miniconda3/envs/graphormer_lora/lib/python3.11/site-packages/transformers/training_args.py:2046: UserWarning: `use_mps_device` is deprecated and will be removed in version 5.0 of 🤗 Transformers. `mps` device will be used by default if available 

GraphormerForGraphClassification(
  (encoder): GraphormerModel(
    (graph_encoder): GraphormerGraphEncoder(
      (dropout_module): Dropout(p=0.0, inplace=False)
      (graph_node_feature): GraphormerGraphNodeFeature(
        (atom_encoder): Embedding(4609, 768, padding_idx=0)
        (in_degree_encoder): Embedding(512, 768, padding_idx=0)
        (out_degree_encoder): Embedding(512, 768, padding_idx=0)
        (graph_token): Embedding(1, 768)
      )
      (graph_attn_bias): GraphormerGraphAttnBias(
        (edge_encoder): Embedding(1537, 32, padding_idx=0)
        (edge_dis_encoder): Embedding(131072, 1)
        (spatial_pos_encoder): Embedding(512, 32, padding_idx=0)
        (graph_token_virtual_distance): Embedding(1, 32)
      )
      (emb_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (layers): ModuleList(
        (0-11): 12 x GraphormerGraphEncoderLayer(
          (dropout_module): Dropout(p=0.0, inplace=False)
          (activation_dropout_module): Dr

In [3]:
train = dataset["train"].with_format("numpy")
valid = dataset["validation"].with_format("numpy")
test = dataset["test"].with_format("numpy")

In [4]:
# Fix for PyTorch 2.6+ checkpoint loading with numpy objects
import torch

# Add numpy globals to safe globals list for checkpoint loading
# This includes numpy array types and all numpy dtype classes
torch.serialization.add_safe_globals([
    np.core.multiarray._reconstruct,
    np.ndarray,
    np.dtype,
    np.core.multiarray.scalar,
    # Add all numpy dtype classes
    np.dtypes.UInt32DType,
    np.dtypes.Int64DType,
    np.dtypes.Float64DType,
    np.dtypes.Float32DType,
    np.dtypes.Int32DType,
    np.dtypes.BoolDType,
    np.dtypes.ObjectDType,
])


/var/folders/qb/7wr1ckc131bbf7qtmmsjfdvw0000gr/T/ipykernel_37901/3610123139.py:7: DeprecationWarning: numpy.core is deprecated and has been renamed to numpy._core. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.multiarray.
  np.core.multiarray._reconstruct,
/var/folders/qb/7wr1ckc131bbf7qtmmsjfdvw0000gr/T/ipykernel_37901/3610123139.py:10: DeprecationWarning: numpy.core is deprecated and has been renamed to numpy._core. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access func

In [5]:
from typing import Dict
import evaluate
from transformers import EvalPrediction
from transformers.models.graphormer.collating_graphormer import GraphormerDataCollator
from trainer import Trainer
roc_auc = evaluate.load("roc_auc")

def compute_metrics(eval_pred: EvalPrediction) -> Dict:
    logits, labels = eval_pred.predictions, eval_pred.label_ids.reshape(-1)
    predictions = np.argmax(logits, axis=-1)
    predictions = np.reshape(predictions, -1)
    result = roc_auc.compute(prediction_scores=predictions, references=labels)
    return result

## Base pretrained model

In [6]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train,
    eval_dataset=valid,
    compute_metrics=compute_metrics,
    data_collator = GraphormerDataCollator(on_the_fly_processing=True),
)

evaluation = trainer.evaluate(eval_dataset=valid)
print(evaluation)

/opt/miniconda3/envs/graphormer_lora/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
100%|██████████| 257/257 [01:28<00:00,  2.91it/s]

{'eval_loss': 0.9538129973968179, 'eval_roc_auc': 0.5020719056938356, 'eval_runtime': 92.2724, 'eval_samples_per_second': 44.575, 'eval_steps_per_second': 2.796}


In [9]:
trained_model = GraphormerForGraphClassification.from_pretrained(
    "./graph-classification/full/molhiv/checkpoint-514"
)

evaluator = Trainer(
    model=trained_model,
    args=training_args,
    train_dataset=train,
    eval_dataset=valid,
    compute_metrics=compute_metrics,
    data_collator = GraphormerDataCollator(on_the_fly_processing=True),
)
valid_evaluation = evaluator.evaluate(eval_dataset=valid)
print(valid_evaluation)

/opt/miniconda3/envs/graphormer_lora/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
100%|██████████| 257/257 [02:09<00:00,  1.98it/s]

{'eval_loss': 0.07863501255701257, 'eval_roc_auc': 0.5491346386492338, 'eval_runtime': 137.3438, 'eval_samples_per_second': 29.947, 'eval_steps_per_second': 1.878}


In [7]:
test_evaluation = evaluator.evaluate(eval_dataset=test)
print(test_evaluation)

/opt/miniconda3/envs/graphormer_lora/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
100%|██████████| 257/257 [01:35<00:00,  2.71it/s]

{'eval_loss': 0.1206226675564619, 'eval_roc_auc': 0.6403778541900089, 'eval_runtime': 98.9341, 'eval_samples_per_second': 41.573, 'eval_steps_per_second': 2.608}
